# The logging schema

Every field the study records, one row per column, for the appendix. The source
of truth is the DDL under `supabase/migrations/2026091*_study_*.sql` …
`20260930120000_study_participant_scores.sql`; the field list it implements is
`docs/event-model.md`. The last cell parses the migrations and fails if a column
exists there that this table does not list, or the other way round, so the
appendix cannot drift from the database.

**Round:** *P* participant round only · *S* simulation only · *P+S* both (the
simulation drives the same application, so its sessions and turns land in the
same tables with `run_type = 'simulation'`).
**Null:** *required* — always set (NOT NULL) · *nullable* — may be empty · *anon* —
always set while the study runs, nullable only so the eight-month anonymisation job
can clear it (D91). Whether a column is actually written is in the *meaning* column.

No data is read here: the notebook needs no database and its output holds no
participant rows.

In [1]:
import pandas as pd
from IPython.display import HTML, display

# (table, level, round, writer, purpose)
TABLES = [
    ("study_sessions", "Session", "P+S", "study API on consent / decline (A12)",
     "One row per consent event or decline; run_type separates pilot, study and simulation."),
    ("study_survey_demographics", "Survey", "P", "study API on submit (A08, A12)",
     "Demographic questionnaire, one row per session (§4.2.2; scales D18)."),
    ("study_survey_post_task", "Survey", "P", "study API on submit or dismiss (A09, A12)",
     "Post-task instrument after the third logged turn; absent = never reached."),
    ("study_suggestions", "Suggestion", "P+S", "turn logger, turn start + end (A02, A03)",
     "One row per agent turn — the comparison unit (D08)."),
    ("study_user_actions", "Suggestion", "P+S", "accept / reject / revert handlers (A05)",
     "Every explicit reaction to a suggestion, one row per target (D12)."),
    ("study_feedback", "Suggestion", "P+S", "thumbs control via study API (A06, A12)",
     "Direct feedback: thumbs under each Tonic reply (D08)."),
    ("study_mutations", "State mutation", "P+S", "DB triggers on plan tables (A04)",
     "Every write to the plan tables in a study session, with its origin (D13)."),
    ("study_verdicts", "Suggestion", "—", "analysis (never written, D73)",
     "Per-suggestion structural verdicts; kept empty since D73."),
    ("study_runs", "Run", "S", "harness runner (B07, A10)",
     "One baseline or regression execution of the suite (Table 11)."),
    ("study_case_executions", "Test case", "S", "harness runner (B02, B03, B07)",
     "One case run once; n rows per case per run."),
    ("study_structural_verdicts", "Test case", "S", "structural checker (D76)",
     "Checker output per execution and checker version, append-only."),
    ("study_scores", "Test case", "S", "point-wise judge (B09, D71)",
     "Point-wise judge score per execution and judge version, append-only."),
    ("study_comparisons", "Test case", "S", "pair-wise judge (B11, D23, D87)",
     "Pair-wise verdict vs the baseline, both orderings, per judge version."),
    ("study_participant_scores", "Suggestion", "P", "tools/score_participants.py (D110)",
     "P3: point-wise judge score of a participant reply, per opening turn."),
]

# (table, column, type, null, values / constraint, meaning)
COLUMNS = [
    # ── study_sessions
    ("study_sessions", "id", "uuid", "required", "PK", "session_id of the event model"),
    ("study_sessions", "user_id", "uuid", "required", "FK profiles, cascade", "The participant; pseudonymous in analysis"),
    ("study_sessions", "run_type", "text", "required", "pilot | study | simulation", "Set at creation, never inferred later (D15)"),
    ("study_sessions", "consent_version", "text", "nullable", "set iff consented", "Version id of the consent text agreed to"),
    ("study_sessions", "consent_at", "timestamptz", "nullable", "consent xor decline", "Time of consent; gates all logging after it"),
    ("study_sessions", "declined_at", "timestamptz", "nullable", "consent xor decline", "Time of decline"),
    ("study_sessions", "device_type", "text", "nullable", "browser-derived", "Phone vs other; tablets read as desktop (D113)"),
    ("study_sessions", "client_session_id", "text", "anon", "X-Session-Id", "Analytics tab id; joins to usage / UX events"),
    ("study_sessions", "created_at", "timestamptz", "required", "default now()", "Row creation"),
    ("study_sessions", "anonymised_at", "timestamptz", "nullable", "", "When the D91 job stripped links and text"),
    # ── study_survey_demographics
    ("study_survey_demographics", "session_id", "uuid", "required", "PK, FK sessions, cascade", "One row per session"),
    ("study_survey_demographics", "user_id", "uuid", "anon", "FK profiles, cascade", "Respondent"),
    ("study_survey_demographics", "age", "integer", "required", "18 – 120", "Exact age"),
    ("study_survey_demographics", "gender", "text", "required", "woman | man | non_binary | prefer_not_to_say", "Gender"),
    ("study_survey_demographics", "education_level", "text", "required", "compulsory | vocational | high_school | bachelor | master_or_higher", "Highest education"),
    ("study_survey_demographics", "llm_familiarity", "text", "required", "never | rarely | monthly | weekly | daily", "How often the respondent uses LLM tools"),
    ("study_survey_demographics", "submitted_at", "timestamptz", "required", "default now()", "Submission time"),
    # ── study_survey_post_task
    ("study_survey_post_task", "session_id", "uuid", "required", "PK, FK sessions, cascade", "One row per session"),
    ("study_survey_post_task", "user_id", "uuid", "anon", "FK profiles, cascade", "Respondent"),
    ("study_survey_post_task", "umux_lite_capability", "integer", "nullable", "1 – 7", "UMUX-LITE item 1 (capabilities meet requirements)"),
    ("study_survey_post_task", "umux_lite_ease", "integer", "nullable", "1 – 7", "UMUX-LITE item 2 (easy to use)"),
    ("study_survey_post_task", "real_trip", "boolean", "nullable", "", "Task involvement: planning a real trip"),
    ("study_survey_post_task", "likelihood_to_recommend", "integer", "nullable", "1 – 7", "Likelihood to recommend"),
    ("study_survey_post_task", "free_text", "text", "nullable", "≤ 4000 chars", "Optional comment; cleared by D91"),
    ("study_survey_post_task", "submitted_at", "timestamptz", "nullable", "exactly one of", "Submitted …"),
    ("study_survey_post_task", "dismissed_at", "timestamptz", "nullable", "submitted / dismissed", "… or dismissed (partial answers legal)"),
    # ── study_suggestions
    ("study_suggestions", "id", "uuid", "required", "PK", "suggestion_id of the event model"),
    ("study_suggestions", "session_id", "uuid", "required", "FK sessions, cascade", "Joins the turn to its session and run_type"),
    ("study_suggestions", "user_id", "uuid", "anon", "FK profiles, cascade", "Member who sent the turn"),
    ("study_suggestions", "trip_id", "uuid", "anon", "FK trips, set null", "Trip the turn was made on"),
    ("study_suggestions", "run_id", "uuid", "anon", "FK tonic_runs, set null", "conversation_id: the planning run"),
    ("study_suggestions", "turn_index", "integer", "required", "", "Position of the turn in the conversation"),
    ("study_suggestions", "user_turn", "text", "anon", "verbatim copy (D06)", "The member's message, verbatim"),
    ("study_suggestions", "user_message_id", "uuid", "anon", "FK chat_messages, set null", "Chat row of the member's message"),
    ("study_suggestions", "reply_message_id", "uuid", "anon", "FK chat_messages, set null", "Chat row of Tonic's reply (thumbs key)"),
    ("study_suggestions", "agent_intent", "text", "nullable", "find | plan | reschedule", "Slash intent, where one was used"),
    ("study_suggestions", "device_type", "text", "nullable", "X-Device-Type header", "General level: device of the turn"),
    ("study_suggestions", "occurred_at", "timestamptz", "required", "default now()", "General level: turn start"),
    ("study_suggestions", "completed_at", "timestamptz", "nullable", "", "Turn end"),
    ("study_suggestions", "latency_ms", "integer", "nullable", "", "Wall-clock latency of the turn"),
    ("study_suggestions", "llm_calls", "integer", "nullable", "", "Model calls in the turn"),
    ("study_suggestions", "tool_calls", "integer", "nullable", "", "Tool calls in the turn"),
    ("study_suggestions", "tokens_input", "integer", "nullable", "", "Prompt tokens"),
    ("study_suggestions", "tokens_output", "integer", "nullable", "", "Completion tokens"),
    ("study_suggestions", "retry_count", "integer", "required", "default 0", "Provider retries (429 / 5xx) in the turn"),
    ("study_suggestions", "model_version", "text", "nullable", "dated release", "Pinned model id (invariant 2)"),
    ("study_suggestions", "model_released_on", "date", "nullable", "", "Release date of model_version"),
    ("study_suggestions", "tool_call_trace", "jsonb", "nullable", "ordered array, see below", "Every tool call; basis of the P1 signature"),
    ("study_suggestions", "resulting_state_shape", "jsonb", "nullable", "D10, see below", "Compact shape of what the turn changed"),
    ("study_suggestions", "request_type", "text", "nullable", "scheduling | venue_lookup | factual_qa | advisory", "Class from the tools called (D09 precedence)"),
    ("study_suggestions", "request_type_rule_version", "text", "nullable", "", "Version of the D09 rule that assigned it"),
    ("study_suggestions", "error_code", "text", "nullable", "timeout | llm_call_limit | empty_reply | llm_error | persist_failed | other", "Turn-level failure (D11)"),
    ("study_suggestions", "user_action", "text", "nullable", "accepted | rejected | ignored | (user_initiated)", "Indirect feedback; final once set; ignored after 24 h (D17)"),
    ("study_suggestions", "structural_check_outcome", "text", "nullable", "none | detected", "Superseded (D73): never written"),
    # ── study_user_actions
    ("study_user_actions", "id", "uuid", "required", "PK", ""),
    ("study_user_actions", "suggestion_id", "uuid", "required", "FK suggestions, cascade", "The turn reacted to"),
    ("study_user_actions", "target", "text", "required", "slot:<i> | place:<google_place_id> | edits", "What was reacted to"),
    ("study_user_actions", "action", "text", "required", "accepted | rejected", "The reaction (a revert is rejected)"),
    ("study_user_actions", "actor_user_id", "uuid", "anon", "FK profiles, cascade", "Who acted (may be a co-member)"),
    ("study_user_actions", "created_at", "timestamptz", "required", "default now()", "Time of the reaction"),
    # ── study_feedback
    ("study_feedback", "id", "uuid", "required", "PK", ""),
    ("study_feedback", "session_id", "uuid", "required", "FK sessions, cascade", "Session of the rater"),
    ("study_feedback", "user_id", "uuid", "anon", "FK profiles, cascade", "Rater; one rating per member per reply"),
    ("study_feedback", "chat_message_id", "uuid", "anon", "FK chat_messages, set null", "The reply rated = suggestions.reply_message_id"),
    ("study_feedback", "value", "text", "required", "up | down", "thumb_feedback"),
    ("study_feedback", "created_at", "timestamptz", "required", "default now()", "First rating"),
    ("study_feedback", "updated_at", "timestamptz", "required", "trigger", "Last change of mind"),
    # ── study_mutations
    ("study_mutations", "id", "uuid", "required", "PK", "mutation_id"),
    ("study_mutations", "table_name", "text", "required", "trip_blocks | itinerary_slots | itinerary_days | ai_suggestions | branches", "Plan table written"),
    ("study_mutations", "row_id", "uuid", "anon", "", "Row written"),
    ("study_mutations", "op", "text", "required", "INSERT | UPDATE | DELETE", "Kind of write (no-op updates filtered)"),
    ("study_mutations", "changed_keys", "text[]", "nullable", "", "Columns changed by an UPDATE"),
    ("study_mutations", "projection", "jsonb", "nullable", "per table, see below", "Structural columns only — never titles or notes"),
    ("study_mutations", "origin_suggestion_id", "uuid", "nullable", "FK suggestions, set null", "The turn that produced the write"),
    ("study_mutations", "origin_kind", "text", "required", "agent_live_edit | agent_draft_accepted | agent_place_added | user_manual | system", "Who / what originated it (x-study-origin header)"),
    ("study_mutations", "actor_user_id", "uuid", "anon", "FK profiles, cascade", "Member whose request wrote it"),
    ("study_mutations", "session_id", "uuid", "nullable", "FK sessions, cascade", "Consented session it belongs to"),
    ("study_mutations", "trip_id", "uuid", "anon", "FK trips, set null", "Trip written"),
    ("study_mutations", "occurred_at", "timestamptz", "required", "default now()", "Time of the write"),
    # ── study_verdicts
    ("study_verdicts", "id", "uuid", "required", "PK", ""),
    ("study_verdicts", "suggestion_id", "uuid", "required", "FK suggestions, cascade", ""),
    ("study_verdicts", "checker_version", "text", "required", "unique with suggestion", ""),
    ("study_verdicts", "outcome", "text", "required", "none | detected", ""),
    ("study_verdicts", "findings", "jsonb", "required", "default []", ""),
    ("study_verdicts", "created_at", "timestamptz", "required", "default now()", ""),
    # ── study_runs
    ("study_runs", "id", "uuid", "required", "PK", "run_id"),
    ("study_runs", "started_at", "timestamptz", "required", "default now()", "Run start"),
    ("study_runs", "finished_at", "timestamptz", "nullable", "", "Run end"),
    ("study_runs", "model_version", "text", "required", "dated release", "Agent model (invariant 2)"),
    ("study_runs", "model_released_on", "date", "nullable", "", "Its release date"),
    ("study_runs", "prompt_version", "text", "nullable", "", "Agent prompt version"),
    ("study_runs", "tool_version", "text", "nullable", "", "Agent toolset version"),
    ("study_runs", "git_sha", "text", "nullable", "", "Harness commit"),
    ("study_runs", "api_version", "text", "nullable", "GET /api/study/versions", "API version under test"),
    ("study_runs", "api_commit", "text", "nullable", "", "API commit under test"),
    ("study_runs", "app_version", "text", "nullable", "", "Application version under test"),
    ("study_runs", "suite_version", "text", "required", "", "Regression suite version"),
    ("study_runs", "baseline_ref", "uuid", "nullable", "FK runs, set null", "Run compared against; null for the baseline"),
    ("study_runs", "simulator_model", "text", "nullable", "", "Simulated-member model"),
    ("study_runs", "simulator_prompt_version", "text", "nullable", "", "Simulator prompt version"),
    ("study_runs", "judge_model", "text", "nullable", "", "Judge model"),
    ("study_runs", "judge_prompt_version", "text", "nullable", "", "Judge prompt version"),
    ("study_runs", "cost_usd", "numeric(12,6)", "nullable", "", "Estimated cost of the run"),
    ("study_runs", "latency_ms", "integer", "nullable", "", "Total latency"),
    ("study_runs", "token_count", "integer", "nullable", "", "Total tokens"),
    ("study_runs", "notes", "text", "nullable", "", "Free-text run note"),
    # ── study_case_executions
    ("study_case_executions", "id", "uuid", "required", "PK", "execution_id"),
    ("study_case_executions", "run_id", "uuid", "required", "FK runs, cascade", "Run it belongs to"),
    ("study_case_executions", "case_id", "text", "required", "", "Test case id"),
    ("study_case_executions", "case_version", "text", "nullable", "", "Hash of the case text executed (D74)"),
    ("study_case_executions", "base_case_id", "text", "nullable", "", "Seed it was derived from; null for a seed"),
    ("study_case_executions", "provenance", "text", "required", "seeded | metamorphic | retained", "Origin of the case"),
    ("study_case_executions", "run_priority", "integer", "nullable", "", "Run priority"),
    ("study_case_executions", "repetition", "integer", "required", "unique with run, case, attempt", "Which of the n repetitions"),
    ("study_case_executions", "repetitions_n", "integer", "nullable", "8 seeds / 4 expanded (D21)", "n for this case"),
    ("study_case_executions", "attempt", "integer", "required", "default 1", "Re-run after a provider refusal (D53)"),
    ("study_case_executions", "session_id", "uuid", "nullable", "FK sessions, set null", "Simulation session used"),
    ("study_case_executions", "trip_id", "uuid", "nullable", "FK trips, set null", "Fixture trip used"),
    ("study_case_executions", "fixture_ids", "jsonb", "nullable", "", "Ids the fixture build produced (D26)"),
    ("study_case_executions", "build_mutation_ids", "uuid[]", "required", "default {}", "Mutations written by the fixture build (D38)"),
    ("study_case_executions", "declared_constraints", "jsonb", "nullable", "", "Constraints the case declares"),
    ("study_case_executions", "expected_tool_sequence", "jsonb", "nullable", "", "Expected tool calls"),
    ("study_case_executions", "scoring_criteria", "jsonb", "nullable", "", "Judge rubric"),
    ("study_case_executions", "suggestion_ids", "uuid[]", "required", "default {}", "Turns produced, in order"),
    ("study_case_executions", "structural_outcome", "text", "nullable", "none | detected", "Checker verdict (mirror of the latest)"),
    ("study_case_executions", "structural_findings", "jsonb", "required", "default []", "Checker findings, ordered strings (D41)"),
    ("study_case_executions", "cost_usd", "numeric(12,6)", "nullable", "", "Cost of the execution"),
    ("study_case_executions", "latency_ms", "integer", "nullable", "", "Latency"),
    ("study_case_executions", "token_count", "integer", "nullable", "", "Tokens"),
    ("study_case_executions", "error", "text", "nullable", "", "Execution-level failure"),
    ("study_case_executions", "started_at", "timestamptz", "required", "default now()", "Start"),
    ("study_case_executions", "finished_at", "timestamptz", "nullable", "", "End"),
    # ── study_structural_verdicts
    ("study_structural_verdicts", "id", "uuid", "required", "PK", ""),
    ("study_structural_verdicts", "execution_id", "uuid", "required", "FK executions, cascade", "Execution checked"),
    ("study_structural_verdicts", "run_id", "uuid", "required", "FK runs, cascade", "Denormalised"),
    ("study_structural_verdicts", "case_id", "text", "required", "", "Denormalised"),
    ("study_structural_verdicts", "repetition", "integer", "required", "", "Denormalised"),
    ("study_structural_verdicts", "attempt", "integer", "required", "default 1", "Denormalised"),
    ("study_structural_verdicts", "checker_version", "text", "required", "unique with execution", "Checker that produced it"),
    ("study_structural_verdicts", "case_version", "text", "nullable", "", "Case text judged (D74)"),
    ("study_structural_verdicts", "outcome", "text", "required", "none | detected", "Verdict"),
    ("study_structural_verdicts", "findings", "jsonb", "required", "default []", "Findings"),
    ("study_structural_verdicts", "created_at", "timestamptz", "required", "default now()", ""),
    # ── study_scores
    ("study_scores", "id", "uuid", "required", "PK", ""),
    ("study_scores", "execution_id", "uuid", "required", "FK executions, cascade", "Execution scored"),
    ("study_scores", "run_id", "uuid", "required", "FK runs, cascade", "Denormalised"),
    ("study_scores", "case_id", "text", "required", "", "Denormalised"),
    ("study_scores", "repetition", "integer", "required", "", "Denormalised"),
    ("study_scores", "attempt", "integer", "required", "default 1", "Attempt scored (highest)"),
    ("study_scores", "judge_model", "text", "required", "unique with execution,", "Judge model"),
    ("study_scores", "judge_prompt_version", "text", "required", "judge_prompt_version", "Judge prompt version"),
    ("study_scores", "score", "integer", "nullable", "1 – 10; score or error", "Point-wise score"),
    ("study_scores", "rationale", "text", "required", "default ''", "Judge's reasoning"),
    ("study_scores", "reply_chars", "integer", "required", "default 0", "Length of the reply scored"),
    ("study_scores", "error", "text", "nullable", "score or error", "Why unscored (missing measurement, D70)"),
    ("study_scores", "created_at", "timestamptz", "required", "default now()", ""),
    # ── study_comparisons
    ("study_comparisons", "id", "uuid", "required", "PK", ""),
    ("study_comparisons", "run_id", "uuid", "required", "FK runs, cascade", "Regression run"),
    ("study_comparisons", "execution_id", "uuid", "required", "FK executions, cascade", "Candidate execution"),
    ("study_comparisons", "baseline_execution_id", "uuid", "required", "FK executions, cascade", "Baseline execution"),
    ("study_comparisons", "case_id", "text", "required", "", "Denormalised"),
    ("study_comparisons", "repetition", "integer", "required", "", "Denormalised"),
    ("study_comparisons", "attempt", "integer", "required", "default 1", "Denormalised"),
    ("study_comparisons", "baseline_kind", "text", "required", "adopted | test", "Adopted baseline vs rehearsal"),
    ("study_comparisons", "baseline_key", "text", "required", "", "Hash of the baseline entries"),
    ("study_comparisons", "judge_model", "text", "required", "unique with both executions,", "Judge model"),
    ("study_comparisons", "judge_prompt_version", "text", "required", "judge_prompt_version", "Judge prompt version"),
    ("study_comparisons", "structural_outcome", "text", "nullable", "snapshot", "Candidate's structural verdict at comparison"),
    ("study_comparisons", "ordering_candidate_b", "text", "nullable", "A | B | tie", "Verdict with candidate shown as B"),
    ("study_comparisons", "ordering_candidate_a", "text", "nullable", "A | B | tie", "Verdict with candidate shown as A"),
    ("study_comparisons", "at_least_as_good", "boolean", "nullable", "decided or error", "Pass: candidate ≥ baseline in both orderings"),
    ("study_comparisons", "rationale", "text", "required", "default ''", "Judge's reasoning"),
    ("study_comparisons", "error", "text", "nullable", "", "Why undecided"),
    ("study_comparisons", "created_at", "timestamptz", "required", "default now()", ""),
    # ── study_participant_scores
    ("study_participant_scores", "id", "uuid", "required", "PK", ""),
    ("study_participant_scores", "suggestion_id", "uuid", "required", "FK suggestions, cascade", "Opening turn of the request"),
    ("study_participant_scores", "session_id", "uuid", "required", "FK sessions, cascade", "Participant session"),
    ("study_participant_scores", "request_type", "text", "required", "scheduling | venue_lookup | factual_qa | advisory", "Rubric used (D09 type)"),
    ("study_participant_scores", "judge_model", "text", "required", "unique with suggestion,", "Judge model"),
    ("study_participant_scores", "judge_prompt_version", "text", "required", "judge_prompt_version", "Judge prompt version"),
    ("study_participant_scores", "score", "integer", "nullable", "1 – 10; score or error", "Point-wise score"),
    ("study_participant_scores", "rationale", "text", "anon", "", "Judge's reasoning; cleared by D91"),
    ("study_participant_scores", "reply_chars", "integer", "required", "default 0", "Length of the reply scored"),
    ("study_participant_scores", "error", "text", "nullable", "score or error", "Why unscored"),
    ("study_participant_scores", "created_at", "timestamptz", "required", "default now()", ""),
]

# Three columns hold a JSON object instead of a single value. A.4 lists what is
# inside them. (group heading, what the column is, [(key, type, meaning)])
P = "copied from the row"
JSON_GROUPS = [
    ("study_suggestions.tool_call_trace — model calls",
     "A list with one entry per step the agent took during the turn, in order. "
     "Entries with kind = 'llm' are calls to the language model:",
     [("kind", "'llm'", "Marks a model call"),
      ("seq", "integer", "Position in the turn (one counter for model and tool entries)"),
      ("model", "text", "Model that answered"),
      ("tokens_input", "integer", "Prompt tokens of this call"),
      ("tokens_output", "integer", "Completion tokens of this call"),
      ("duration_ms", "integer", "How long the call took"),
      ("error", "text, optional", "Provider error, if the call failed")]),
    ("study_suggestions.tool_call_trace — tool calls",
     "Entries with kind = 'tool' are the tools the model called (search_places, propose_plan, …):",
     [("kind", "'tool'", "Marks a tool call"),
      ("seq", "integer", "Position in the turn"),
      ("tool", "text", "Tool name"),
      ("args", "object", "Arguments as sent; values cleared after 8 months, keys kept (D91)"),
      ("started_at", "timestamp", "When the call started"),
      ("duration_ms", "integer", "How long the call took"),
      ("result_error", "text | null", "Error the tool returned instead of a result (D11)"),
      ("result_chars", "integer", "Size of the result; the content itself is not stored"),
      ("result_found", "boolean | null", "Whether a source had an answer (read_wikipedia, get_region_climate only; D40)")]),
    ("study_suggestions.resulting_state_shape",
     "One object per turn summarising what the turn produced and changed — counts, never content (D10):",
     [("draft.suggestion_id", "uuid | null", "Draft plan the turn saved; null if none (cleared after 8 months)"),
      ("draft.days", "integer", "Days in that draft"),
      ("draft.slots", "integer", "Stops in that draft"),
      ("draft.slot_types", "text[]", "Type of each stop (food, sight, …)"),
      ("edits.move", "integer", "Stops the agent moved directly in the itinerary"),
      ("edits.unschedule", "integer", "Stops it took off the schedule"),
      ("edits.schedule", "integer", "Stops it put on the schedule"),
      ("places_surfaced", "integer", "Places returned by search_places"),
      ("reply.chars", "integer", "Length of the reply"),
      ("reply.place_refs", "integer", "Place cards in the reply"),
      ("reply.draft_refs", "integer", "Draft cards in the reply"),
      ("trip_after.blocks", "integer | null", "Stops on the trip after the turn"),
      ("trip_after.slots", "integer | null", "Scheduled slots on the trip after the turn")]),
    ("study_mutations.projection — trip_blocks",
     "A copy of a few structural columns of the row that was written; which ones depends on the table. "
     "Never titles, notes or other text. For a stop (trip_blocks):",
     [("type", P, "Kind of stop"), ("source", P, "How the stop was created (e.g. manual, ai, places)"),
      ("place_id", P, "Linked venue"), ("place_link", P, "auto | linked | unlinked"),
      ("region_id", P, "Region of the trip"), ("trip_leg_id", P, "Leg of the trip")]),
    ("study_mutations.projection — itinerary_slots", "For a scheduled slot:",
     [("itinerary_day_id", P, "Day row"), ("day_date", "date", "The day (looked up from the day row)"),
      ("start_time", P, "Start"), ("end_time", P, "End"), ("end_day_date", P, "Last day of a multi-day stay"),
      ("time_precision", P, "exact | band (a part of the day, not a clock time)"), ("trip_block_id", P, "The stop in the slot"),
      ("is_locked", P, "Pinned by a member")]),
    ("study_mutations.projection — itinerary_days", "For a day:",
     [("day_date", P, "The day")]),
    ("study_mutations.projection — ai_suggestions", "For a draft plan:",
     [("suggestion_type", P, "Kind of draft"), ("status", P, "pending, accepted, rejected, expired, …"),
      ("version", P, "Draft version"), ("plan_thread_id", P, "Planning thread"),
      ("slots", "integer", "Stops in the draft (counted)"),
      ("settled", "integer", "Stops already accepted or rejected (counted)")]),
    ("study_mutations.projection — branches", "For a Plan B / vote alternative:",
     [("itinerary_slot_id", P, "Slot it is an alternative for"), ("trip_block_id", P, "The alternative stop"),
      ("resolution_mode", P, "vote | conditional"), ("is_main", P, "Currently the chosen option"),
      ("is_resolved", P, "Decision settled"), ("conflict_group_id", P, "Options competing for one window")]),
]
JSON_KEYS = [k for _, _, keys in JSON_GROUPS for k in keys]

meta = pd.DataFrame(TABLES, columns=["table", "level", "round", "writer", "purpose"]).set_index("table")
cols = pd.DataFrame(COLUMNS, columns=["table", "column", "type", "null", "values / constraint", "meaning"])
print(f"{len(meta)} tables · {len(cols)} columns · {len(JSON_KEYS)} JSON keys")

14 tables · 185 columns · 56 JSON keys


In [2]:
CSS = """
<style>
.schema { border-collapse: collapse; font: 9.5px/1.25 'Segoe UI', Arial, sans-serif; color: #111;
          background: #fff; width: 1060px; table-layout: fixed; }
.schema th { background: #222; color: #fff; text-align: left; padding: 3px 5px; font-weight: 600; }
.schema td { padding: 1.5px 5px; border-bottom: 1px solid #e3e3e3; vertical-align: top; overflow-wrap: anywhere; }
.schema td.c { font-family: Consolas, 'Courier New', monospace; font-size: 9.5px; }
.schema td.v { font-family: Consolas, 'Courier New', monospace; font-size: 8.8px; color: #333; }
.schema td.n { color: #555; }
.schema tr.grp td { background: #ececec; border-top: 1.5px solid #555; padding: 3px 5px; font-weight: 600; }
.schema tr.grp span { font-weight: 400; color: #444; }
.schema tr.grp code { font-size: 10px; }
.schema caption { caption-side: top; text-align: left; font: 600 11px 'Segoe UI', Arial; padding: 4px 0; }
</style>"""

def esc(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def schema_table(tables, caption):
    rows = []
    for t in tables:
        m = meta.loc[t]
        rows.append(f'<tr class="grp"><td colspan="5"><code>{t}</code> &nbsp;·&nbsp; {m.level} level '
                    f'&nbsp;·&nbsp; {m["round"]} &nbsp;·&nbsp; <span>{esc(m.purpose)} Writer: {esc(m.writer)}.</span></td></tr>')
        for _, r in cols[cols.table == t].iterrows():
            rows.append(f'<tr><td class="c">{esc(r.column)}</td><td class="c">{esc(r.type)}</td>'
                        f'<td class="n">{r.null}</td><td class="v">{esc(r["values / constraint"])}</td>'
                        f'<td>{esc(r.meaning)}</td></tr>')
    head = "<tr><th>column</th><th>type</th><th>null</th><th>values / constraint</th><th>meaning</th></tr>"
    return HTML(f'{CSS}<table class="schema"><caption>{caption}</caption><colgroup><col style="width:175px"><col style="width:95px"><col style="width:62px"><col style="width:330px"><col></colgroup><thead>{head}</thead>'
                f'<tbody>{"".join(rows)}</tbody></table>')

def json_table():
    rows = []
    for title, about, keys in JSON_GROUPS:
        rows.append(f'<tr class="grp"><td colspan="3"><code>{esc(title)}</code><br><span>{esc(about)}</span></td></tr>')
        for key, typ, meaning in keys:
            rows.append(f'<tr><td class="c">{esc(key)}</td><td class="v">{esc(typ)}</td><td>{esc(meaning)}</td></tr>')
    head = "<tr><th>key</th><th>type</th><th>meaning</th></tr>"
    return HTML(f'{CSS}<table class="schema"><caption>Table A.4 — Inside the JSON columns</caption>'
                f'<colgroup><col style="width:175px"><col style="width:160px"><col></colgroup>'
                f'<thead>{head}</thead><tbody>{"".join(rows)}</tbody></table>')

The table is split in three so each part fits a printed page: the participant-facing
levels (session, surveys, suggestion, mutation), the simulation levels (run, test case,
judge and checker output), and the keys inside the JSON columns.

In [3]:
schema_table(["study_sessions", "study_survey_demographics", "study_survey_post_task",
              "study_suggestions"],
             "Table A.1 — Session, survey and suggestion levels")

In [4]:
schema_table(["study_user_actions", "study_feedback", "study_mutations",
              "study_participant_scores", "study_verdicts"],
             "Table A.2 — Feedback, state-mutation and participant-scoring levels")

In [5]:
schema_table(["study_runs", "study_case_executions", "study_structural_verdicts",
              "study_scores", "study_comparisons"],
             "Table A.3 — Run and test-case levels (simulation only)")

Three columns in A.1–A.2 hold a JSON object rather than a single value: `tool_call_trace` (every model and tool call of a turn), `resulting_state_shape` (what the turn produced, as counts) and `projection` (a few columns of the plan row a mutation wrote). A.4 lists the keys inside each.

In [6]:
json_table()

## Check against the migrations

Replays `create table` and `add column` over every study migration and compares the
result with the table above. A mismatch fails here, not in the appendix.

In [7]:
import re, pathlib
MIG = pathlib.Path.cwd().resolve().parents[3] / "supabase" / "migrations"
ddl = {}
for f in sorted(MIG.glob("*.sql")):
    sql = f.read_text(encoding="utf-8").replace("\r\n", "\n")
    for m in re.finditer(r'create table if not exists "public"\."(study_\w+)" \((.*?)\n\);', sql, re.S):
        body = re.sub(r"--[^\n]*", "", m.group(2))
        ddl[m.group(1)] = {c for c in re.findall(r'^\s*"(\w+)"\s+\w', body, re.M)}
    for m in re.finditer(r'alter table "public"\."(study_\w+)"((?:\s*add column if not exists "\w+"[^,;]*,?)+);', sql):
        ddl.setdefault(m.group(1), set()).update(re.findall(r'add column if not exists "(\w+)"', m.group(2)))

listed = cols.groupby("table")["column"].apply(set).to_dict()
assert set(ddl) == set(listed) == set(meta.index), (set(ddl) ^ set(listed))
for t in ddl:
    missing, extra = ddl[t] - listed[t], listed[t] - ddl[t]
    assert not missing and not extra, f"{t}: missing {missing}, not in DDL {extra}"
print(f"OK — {len(ddl)} tables, {sum(map(len, ddl.values()))} columns, all listed")

OK — 14 tables, 185 columns, all listed
